# Gabarito dos Exercícios — Módulo 3
### Curso Introdutório de Python para Ciência de Dados
**Disciplina:** T326 - Ciência de Dados | UNIFOR

> ⚠️ **Atenção:** Este arquivo é o gabarito. Tente resolver os exercícios antes de consultá-lo!

---


In [ ]:
# ── Setup inicial ────────────────────────────────────────────────
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns, io, warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white', 'axes.grid': True, 'grid.alpha': 0.4})
sns.set_palette('Set2')

def carregar_brazilian_cities(caminho):
    with open(caminho, 'r', encoding='utf-8', newline='') as f:
        raw = f.read()
    linhas = raw.split('\r\n')
    def fix(l):
        if l.startswith('"') and l.endswith('"'): l = l[1:-1]
        return l.replace('""', '"')
    return pd.read_csv(io.StringIO('\n'.join([fix(l) for l in linhas if l.strip()])))

df = carregar_brazilian_cities('../datasets/brazilian_city.csv')
df = df.rename(columns={'IDHM Ranking 2010':'IDHM_Ranking','IBGE_CROP_PRODUCTION_$':'IBGE_CROP_PROD',
    'WAL-MART':'WALMART','IBGE_1-4':'IBGE_1a4','IBGE_5-9':'IBGE_5a9',
    'IBGE_10-14':'IBGE_10a14','IBGE_15-59':'IBGE_15a59','IBGE_60+':'IBGE_60mais'})
df['POP_FINAL'] = df['IBGE_RES_POP'].where(df['IBGE_RES_POP'] > 0, df['ESTIMATED_POP'])
df['DENSIDADE'] = np.where(df['AREA'] > 0, (df['POP_FINAL']/df['AREA']).round(2), np.nan)
df['TIPO'] = df['CAPITAL'].map({1:'Capital', 0:'Interior'})
df['IDHM_FAIXA'] = pd.cut(df['IDHM'], bins=[0,.499,.599,.699,.799,1.],
    labels=['Muito Baixo','Baixo','Médio','Alto','Muito Alto'])
regioes = {'Norte':['AM','PA','RR','RO','AC','AP','TO'],
           'Nordeste':['MA','PI','CE','RN','PB','PE','AL','SE','BA'],
           'Centro-Oeste':['MT','MS','GO','DF'],
           'Sudeste':['SP','RJ','MG','ES'],
           'Sul':['PR','SC','RS']}
df['REGIAO'] = df['STATE'].map({uf:reg for reg,ufs in regioes.items() for uf in ufs})
print("✅ Dataset carregado!")


---
## Gabarito — Módulo 3

### Exercício 1 — Histograma PIB per Capita

In [ ]:
# ── Gabarito Exercício 3.1 ──────────────────────────────────────
gdp = df[(df['GDP_CAPITA'] > 0) & (df['GDP_CAPITA'] < 150_000)]['GDP_CAPITA']

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(gdp, bins=50, color='#4C72B0', edgecolor='white', alpha=0.85)
ax.axvline(gdp.mean(),   color='red',   linestyle='--', lw=2, label=f'Média = R${gdp.mean():,.0f}')
ax.axvline(gdp.median(), color='green', linestyle='-.', lw=2, label=f'Mediana = R${gdp.median():,.0f}')
ax.set_title('Distribuição do PIB per Capita nos Municípios Brasileiros', fontsize=13, fontweight='bold')
ax.set_xlabel('PIB per Capita (R$)'); ax.set_ylabel('Municípios')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
ax.legend()
plt.tight_layout(); plt.show()


### Exercício 2 — 10 Cidades mais Populosas

In [ ]:
# ── Gabarito Exercício 3.2 ──────────────────────────────────────
top10 = df[['CITY','STATE','POP_FINAL']].sort_values('POP_FINAL', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10['CITY'] + ' - ' + top10['STATE'], top10['POP_FINAL'],
               color='#4C72B0', edgecolor='white')
for bar, val in zip(bars, top10['POP_FINAL']):
    ax.text(val + 50_000, bar.get_y() + bar.get_height()/2,
            f'{val/1e6:.2f}M', va='center', fontsize=9, fontweight='bold')
ax.set_title('10 Municípios Mais Populosos do Brasil', fontsize=13, fontweight='bold')
ax.set_xlabel('População'); ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout(); plt.show()


### Exercício 3 — Boxplot GDP_CAPITA por Zona

In [ ]:
# ── Gabarito Exercício 3.3 ──────────────────────────────────────
df_box = df[(df['GDP_CAPITA'] > 0) & (df['GDP_CAPITA'] < 200_000)]
zonas = ['Urbano', 'Rural Adjacente', 'Rural Remoto']
df_box = df_box[df_box['RURAL_URBAN'].isin(zonas)]

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df_box, x='RURAL_URBAN', y='GDP_CAPITA', order=zonas,
            palette='Set2', width=0.5, fliersize=2, ax=ax)
ax.set_title('PIB per Capita por Classificação Urbana/Rural', fontsize=13, fontweight='bold')
ax.set_xlabel('Classificação'); ax.set_ylabel('PIB per Capita (R$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
plt.tight_layout(); plt.show()


### Exercício 4 — Scatter Área × População (log)

In [ ]:
# ── Gabarito Exercício 3.4 ──────────────────────────────────────
df_sc = df[(df['POP_FINAL']>100) & (df['AREA']>0) & df['REGIAO'].notna()].copy()
ordem = ['Norte','Nordeste','Centro-Oeste','Sudeste','Sul']
palette = dict(zip(ordem, ['#E07B54','#E0C354','#54A0E0','#54E089','#9B54E0']))

fig, ax = plt.subplots(figsize=(10, 6))
for reg in ordem:
    sub = df_sc[df_sc['REGIAO']==reg]
    ax.scatter(sub['AREA'], sub['POP_FINAL'], alpha=0.4, s=15,
               color=palette[reg], label=reg)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_title('Área × População dos Municípios Brasileiros (escala log)', fontsize=13, fontweight='bold')
ax.set_xlabel('Área (km²)'); ax.set_ylabel('População')
ax.legend(title='Região', framealpha=0.9)
plt.tight_layout(); plt.show()


### Exercício 5 — Visualização Livre (exemplo)

In [ ]:
# ── Gabarito Exercício 3.5 — Exemplo: TV por Assinatura × IDHM ──
df_tv = df[(df['PAY_TV'] > 0) & (df['POP_FINAL'] > 0)].copy()
df_tv['PAY_TV_PER_100'] = (df_tv['PAY_TV'] / df_tv['POP_FINAL'] * 100).round(2)
df_tv = df_tv[df_tv['PAY_TV_PER_100'] < 50]

fig, ax = plt.subplots(figsize=(10, 6))
for reg in ordem:
    sub = df_tv[df_tv['REGIAO']==reg]
    ax.scatter(sub['IDHM'], sub['PAY_TV_PER_100'], alpha=0.5, s=20,
               color=palette[reg], label=reg)

ax.set_title('IDHM × Penetração de TV por Assinatura (por 100 habitantes)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('IDHM'); ax.set_ylabel('Assinantes de TV Paga por 100 hab.')
ax.legend(title='Região')

# Linha de tendência
z = np.polyfit(df_tv['IDHM'], df_tv['PAY_TV_PER_100'], 1)
x_l = np.linspace(df_tv['IDHM'].min(), df_tv['IDHM'].max(), 100)
ax.plot(x_l, np.poly1d(z)(x_l), 'k--', alpha=0.7, lw=2, label='Tendência')
ax.legend(title='Região')
plt.tight_layout(); plt.show()

corr = df_tv[['IDHM','PAY_TV_PER_100']].corr().iloc[0,1]
print(f"Correlação IDHM × TV por Assinatura: {corr:.3f}")
print("Interpretação: quanto maior o IDHM, maior a penetração de TV a cabo.")
